In [1]:
from py_module.metadata.connection.ConnectionRegistry import PostgresConnection as pgconn
from py_module.metadata.render_jinja.RenderRegistry import ChangeDataCaptureExtraction as cdc, RetrieveSchema as rs, SchemaSource as ss 
from py_module.metadata.connection.StorageBaseConnection import StorageConn as storage
from py_module.metadata.datatype_conversion.avro import DataTypeConverter as dtc
from py_module.exec.SourceToAvro import ExecuteGCS
from sqlalchemy import text
import json
from fastavro import writer, parse_schema
from datetime import datetime as dt, timedelta


# SETUP (connection + get template)

In [2]:
params = {
    "database":"postgres",
    "db": "meteo",
    "host": "localhost",
    "port": 5433,
    "user": "meteo",
    "password": "meteo",
    "ssl_args": {}
} 

In [3]:
engine = pgconn.get_engine(**params)
conn = engine.connect()

2025-10-26 09:33:40 - [INFO   ] | PostgreSQL engine created successfully


In [27]:
cdc_extraction = cdc.render_jinja(database=params['database'])
retrieve_schema = rs.render_jinja(database=params['database'])
schema_source = ss.render_jinja()

# Render template with appropriate values

### Retrieve schema table -- this is useful for next steps

In [32]:
table_schema = 'public'
table_name = None
rendered_schema = retrieve_schema.render(
                                table_schema=table_schema,
                                table_name=table_name
                            )

print(rendered_schema)

select distinct
    c.table_name,
    c.table_schema,
    c.column_name as col_name,
    c.data_type as data_type,
    c.numeric_precision as precision,
    c.numeric_scale as scale
from information_schema.columns c
    inner join information_schema.tables t
        on c.table_schema || c.table_name = t.table_schema || c.table_name 
where true 
    and lower(t.table_type) like '%table%'
    
     and c.table_schema = 'public'


In [33]:
it_schema = conn.execute(text(rendered_schema))

In [34]:
tables = it_schema.fetchall()

In [35]:
import pandas as pd
pd.DataFrame(tables)

,table_name,table_schema,col_name,data_type,precision,scale
0,fct_meteo,public,apparent_temperature,numeric,NaN,NaN
1,fct_meteo,public,city,text,NaN,NaN
2,fct_meteo,public,cloud_cover,numeric,NaN,NaN
3,fct_meteo,public,dew_point,numeric,NaN,NaN
4,fct_meteo,public,insert_timestamp,timestamp without time zone,NaN,NaN
...,...,...,...,...,...,...
95,work_meteo,public,windb_earing,numeric,NaN,NaN
96,work_missing,public,city,text,NaN,NaN
97,work_missing,public,province,text,NaN,NaN
98,work_missing,public,province_code,text,NaN,NaN


In [ ]:
converter = dtc()  # your DataTypeConverter instance, optionally pass defaults

dbt_columns = list()
avro_columns = list()
cdc_columns = list()

for i in it_schema:
    col_name = i[0]
    db_type = i[1]
    precision = i[2]
    scale = i[3]

    avro_type = converter.source_to_avro(params['database'], db_type, numeric_precision=precision, numeric_scale=scale)
    bq_type = converter.source_to_bigquery(params['database'], db_type)

    # setup for cdc model injection
    cdc_columns.append(col_name)

    # setup the columns for avro injection with default
    avro_field = converter.generate_avro_field(col_name, avro_type)
    avro_columns.append(avro_field)

    # setup the columns for dbt source
    dbt_columns.append({col_name: bq_type})

TypeError: PostgresConnection.get_engine() takes 1 positional argument but 2 were given